In [ ]:
import os
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, last, min as spark_min

bucket = os.environ["PNAD_GCS_BUCKET"]
dataset = os.environ["PNAD_BQ_DATASET"]
lake = f"gs://{bucket}/pnad"
spark = DataprocSparkSession.builder.getOrCreate()

In [ ]:
#P1---Extrai dicionario da camada raw e carrega na camada bronze
df_dic = spark.read.csv(f"{lake}/raw/dicionario/", header=True)
df_dic.write.mode("overwrite").option("header", True).parquet(f"{lake}/bronze/dicionario/")

#Dataframe p/ proxima etapa
df_dic_bronze = df_dic

In [ ]:
#P2---Transformacao do dicionario
#Altera "Id" para int
df_dic_bronze = df_dic_bronze.withColumn("id", col("id").cast("int"))

#Filtra campo "Seletor" com os dados pre-selecionados
df_dic_bronze = df_dic_bronze.filter(df_dic_bronze["seletor"] == "OK")

#Preenche campos nulos abaixo com valores acima
cols = ["Variavel", "Codigo Variavel", "Descricao Variavel"]
window = Window.orderBy("Id").rowsBetween(Window.unboundedPreceding, 0)
for c in cols:
    df_dic_bronze = df_dic_bronze.withColumn(c, last(col(c), ignorenulls=True).over(window))

#Renomeia colunas com descricao e letras minusculas
mapeamento = {
    "id": "id",
    "Seletor": "seletor",
    "Variavel": "variavel",
    "Codigo Variavel": "codigo_variavel",
    "Descricao Variavel": "descricao_variavel",
    "Cod Referencia": "cod_referencia",
    "Descricao Referencia": "descricao_referencia",
    "Categoria": "categoria"
}

df_rename = df_dic_bronze
for old, new in mapeamento.items():
    df_rename = df_rename.withColumnRenamed(old, new)
df_dic_bronze = df_rename

In [ ]:
#P3---Carrega dicionario na camada silver
df_dic_bronze.write.mode("overwrite").option("header", True).parquet(f"{lake}/silver/dicionario/")

#Dataframe p/ proxima etapa
df_dic_silver = df_dic_bronze
df_dic_silver = df_dic_silver.orderBy("Id")

In [ ]:
#P1---Extrai arquivos da camada raw e carrega da camada bronze
df_dados = spark.read.csv(f"{lake}/raw/pesquisa/", header=True)
df_dados.write.mode("overwrite").option("header", True).parquet(f"{lake}/bronze/pesquisa/")
# df_dados.select("V1013").distinct().show()

#Dataframe p/ proxima etapa
df_dados_bronze = df_dados

In [ ]:
#P3---Transformacao dos dados
#Filtra colunas contigas no dicionario pre-selecionadas
remover = ["A006B"]
variaveis = [
    row["variavel"]
    for row in df_dic_silver.groupBy("variavel").agg(spark_min("id").alias("ordem")).orderBy("ordem").collect()
    if row["variavel"] not in remover
]
df_dados_bronze = df_dados_bronze.select(variaveis)

#Substituicao de codigos numericos por descricao
df_dic = df_dic_silver.select(
  "variavel",
  "cod_referencia",
  "descricao_referencia"
)

df_result = df_dados_bronze
remover = ["Ano","V1013","A002","A006B","C01012"]
variaveis = [
    row["variavel"]
    for row in df_dic_silver.select("variavel").distinct().collect()
    if row["variavel"] not in remover
]

for var in variaveis:  # lista de colunas
    df_lookup = df_dic.filter(col("variavel") == var) \
        .select(
            col("cod_referencia").alias(f"{var}_cod"),
            col("descricao_referencia").alias(f"{var}_desc")
        )

    df_result = df_result.join(
        df_lookup,
        df_result[var] == df_lookup[f"{var}_cod"],
        "left"
    ).drop(f"{var}_cod") \
     .withColumn(var, col(f"{var}_desc")) \
     .drop(f"{var}_desc")
df_dados_bronze = df_result

#Renomeia colunas com descricao e letras minusculas
#Variaveis
mapeamento = {
    "E001": "emprestimo_tomado",
    "UF": "uf",
    "B008": "teste_covid",
    "B009B": "resultado_exame_swab",
    "CAPITAL": "capital",
    "A003": "sexo",
    "B009A": "teste_swab_realizado",
    "B0031": "ficou_em_casa",
    "C013": "trabalho_remoto",
    "B0011": "sintoma_febre",
    "C007D": "atividade_empresa",
    "B009E": "teste_sangue_veia",
    "B002": "estabelecimento_saude",
    "A006": "frequenta_escola",
    "A006B": "aula_presencial",
    "V1013": "mes_pesquisa",
    "B0014": "sintoma_dificuldade_respirar",
    "B009C": "teste_sangue_dedo",
    "A005": "escolaridade",
    "B009F": "resultado_exame_sangue_veia",
    "B0012": "sintoma_tosse",
    "B009D": "resultado_exame_sangue_dedo",
    "Ano": "ano",
    "B00111": "sintoma_perda_olfato_paladar",
    "A002": "idade",
    "C01012": "valor_dinheiro",
    "B011": "isolamento_social",
    "V1022": "situacao_domicilio"
}

df_rename = df_dados_bronze
for old, new in mapeamento.items():
    df_rename = df_rename.withColumnRenamed(old, new)
df_dados_bronze = df_rename

In [ ]:
#P3---Carrega dados na camada silver
df_dados_bronze.write.mode("overwrite").option("header", True).parquet(f"{lake}/silver/pesquisa/")

#Dataframe p/ proxima etapa
df_dados_silver = df_dados_bronze

In [ ]:
# df_dados_silver = df_dados_silver.select([
#     col(c).cast("string").alias(c) for c in df_dados_silver.columns
# ])

df_dados_silver.write.format("bigquery") \
    .option("table", f"{dataset}.f_pesquisa_covid") \
    .option("temporaryBucket", bucket) \
    .mode("overwrite") \
    .save()

In [ ]:
df_dic_silver = df_dic_silver.select([
    col(c).cast("string").alias(c) for c in df_dic_silver.columns
])

df_dic_silver.write.format("bigquery") \
    .option("table", f"{dataset}.d_dicionario_covid") \
    .option("temporaryBucket", bucket) \
    .mode("overwrite") \
    .save()